# 04 · 계면 거리 분석 — SRO/MRO 의 공간·온도 변화

세 가지 목표:
1. **벌크·계면 RDF** (거리 양끝 = 벌크/계면)
2. **계면 영역 폭 vs 온도** (계면이 얼마나 줄어드나)
3. **계면으로부터 거리별 RDF·FSDP 변화** (SRO=1st peak, MRO=FSDP·2nd shell), 온도별로

방법: 각 온도 4D → **구조맵으로 계면 위치·폭** 파악 → **계면으로부터 거리 밴드별로 패턴 평균 → RDF** →
1st peak(SRO)·FSDP/2nd shell(MRO)를 거리·온도의 함수로.
> 계면이 **세로**라고 가정(구조맵 x-프로파일의 피크=계면 열). 기운 계면이면 per-row로 바꾸세요.

In [ ]:
import os, glob
import numpy as np
import matplotlib.pyplot as plt
import fourdstem as fds

DATA_ROOT = "/home/jonghoonk918/Desktop/fdstem/Amorphous/In-situ/Heating-SiO"
USE_SYNTHETIC = not os.path.isdir(DATA_ROOT)

DET_BIN        = 4          # 검출기 비닝(메모리). 합성이면 1
Q_UNIT_HINT    = "1/A"
Q_PER_PX       = 0.0120     # nb1 2b 캘리브레이션 값 (비닝하면 ×DET_BIN 자동)
BEAM_RADIUS_PX = 12
SCAN_STEP_NM   = None       # 스캔 픽셀 크기(nm). 알면 거리 nm로 표시, 모르면 px
DIST_BINS_PX   = [0, 1, 2, 3, 5, 7, 10, 14]     # 계면으로부터 거리 밴드(스캔 px)
CFG = fds.RDFConfig(composition={"Si": 1, "O": 2}, q_int_min=0.20, q_int_max=1.50,
                    r_min=1.10, r_max=8.0, dr=0.02, damping="lorch")
FSDP_WIN = (0.40, 0.95); SECOND_WIN = (2.3, 3.4); FIRST_WIN = (1.45, 1.85)
xunit = "nm" if SCAN_STEP_NM else "px"; xscale = SCAN_STEP_NM or 1.0
print("USE_SYNTHETIC =", USE_SYNTHETIC)

## 헬퍼 — 큐브 로드 & 계면 위치/거리맵

`interface_distance(cube)` → 구조맵 `s`, 계면 열 `x_if`, **계면 폭(width)**, **거리맵 `dmap`**(각 위치가
계면에서 몇 px).

In [ ]:
def make_gradient_cube(scan=(50, 60), dp=(64, 64), iface_col=30, lam=6.0,
                       bulk_r=26.0, iface_r=16.0, seed=0):
    '''계면(iface_col)에서 거리에 따라 벌크↔계면 구조가 섞이는 데모. lam=영향 감쇠길이(온도↑→lam↓).
    계면=안쪽 링(iface_r, 구조맵 밝음), 벌크=바깥 링(bulk_r) → 계면이 밝은 세로 띠로 잡힘.'''
    rng = np.random.default_rng(seed); Sy, Sx = scan; H, W = dp
    yy, xx = np.mgrid[0:H, 0:W]; cx, cy = W/2, H/2; rr = np.hypot(xx-cx, yy-cy)
    bulk  = np.exp(-(rr-bulk_r)**2/(2*3.0**2)) + 3*np.exp(-rr**2/(2*2**2))
    iface = np.exp(-(rr-iface_r)**2/(2*3.0**2)) + 3*np.exp(-rr**2/(2*2**2))
    cube = np.empty((Sy, Sx, H, W), np.float32)
    for ix in range(Sx):
        w = np.exp(-abs(ix - iface_col)/lam)         # 계면 성분 가중(거리↑→0)
        cube[:, ix] = ((1-w)*bulk + w*iface) + 0.02*rng.standard_normal((Sy, H, W))
    return np.clip(cube, 0, None)

def load_cube(path):
    cb = fds.load(path, Q_UNIT_HINT)
    if DET_BIN > 1: cb = fds.bin_cube_detector(cb, DET_BIN)
    return cb

def interface_distance(cube, smooth=3):
    md_ = cube.mean_dp(); (ccx, ccy), _ = fds.find_center(md_, fds.beam_stopper_mask(md_))
    s = fds.structural_map(cube, center=(ccx, ccy))       # (Sy, Sx)
    prof = s.mean(0)                                        # x-프로파일 (세로 계면 → 여기 피크)
    if smooth > 1:
        prof = np.convolve(prof, np.ones(smooth)/smooth, mode="same")
    base = np.median(prof); peak = prof.max(); x_if = int(np.argmax(prof))
    half = base + 0.5*(peak - base)
    above = np.where(prof >= half)[0]
    contrast = float(peak - base)
    width = float(above.max() - above.min() + 1) if above.size and contrast > 1e-4 else 0.0
    Sx = s.shape[1]
    dmap = np.abs(np.arange(Sx)[None, :] - x_if) + 0.0*s   # 거리맵(스캔 px)
    return dict(s=s, prof=prof, x_if=x_if, width=width, contrast=contrast,
                dmap=dmap, center=(ccx, ccy))

# 데모/실데이터에서 기준 온도 하나 로드
if USE_SYNTHETIC:
    cube = fds.from_array(make_gradient_cube(lam=6.0, seed=300), q_per_px=Q_PER_PX)
else:
    cube = load_cube(sorted(glob.glob(os.path.join(DATA_ROOT, "*.dm4")))[0])
qpp = cube.calibration.q_per_px or Q_PER_PX
info = interface_distance(cube)
print(f"interface column x_if={info['x_if']}, width={info['width']:.1f} px, contrast={info['contrast']:.4f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
im = ax[0].imshow(info["s"], cmap="viridis"); ax[0].axvline(info["x_if"], color="r", ls="--")
ax[0].set_title("structural map (+ interface line)"); ax[0].axis("off")
ax[1].plot(info["prof"]); ax[1].axvline(info["x_if"], color="r", ls="--")
ax[1].set_xlabel("scan x (px)"); ax[1].set_ylabel("structural contrast"); ax[1].set_title("x-profile → interface & width")
plt.tight_layout(); plt.show()

## 1) 거리별 RDF — 벌크·계면 & 거리에 따른 SRO/MRO 변화 (목표 1·3, 기준 온도)

계면으로부터 **거리 밴드별로 패턴을 평균 → RDF**. 가장 가까운 밴드=계면상, 먼 밴드=벌크상. 각 밴드에서
**1st peak(SRO), FSDP·2nd shell(MRO)** 를 뽑아 거리의 함수로 봅니다.

In [ ]:
def distance_profiles(cube, info):
    dmap = info["dmap"]; center = info["center"]
    out = []
    for i in range(len(DIST_BINS_PX)-1):
        band = (dmap >= DIST_BINS_PX[i]) & (dmap < DIST_BINS_PX[i+1])
        if band.sum() < 5: continue
        pat = fds.average_pattern(cube, band)
        rr = fds.pattern_to_rdf(pat, qpp, CFG, center=center,
                                center_beam_radius=max(1, BEAM_RADIUS_PX//max(DET_BIN, 1)))
        d = 0.5*(DIST_BINS_PX[i]+DIST_BINS_PX[i+1]) * xscale
        r1, _   = fds.first_peak_position(rr.r, rr.Gr, *FIRST_WIN)
        r2      = fds.peak_centroid(rr.r, rr.Gr, *SECOND_WIN)
        qf, _   = fds.first_peak_position(rr.q_reduced, rr.phi, *FSDP_WIN)
        out.append(dict(dist=d, rr=rr, r1=r1, r2=r2, fsdp=qf, n=int(band.sum())))
    return out

prof = distance_profiles(cube, info)
dists = np.array([p["dist"] for p in prof])
cm = plt.get_cmap("plasma"); norm = plt.Normalize(dists.min(), dists.max())

fig, ax = plt.subplots(1, 3, figsize=(15, 4.3))
for p in prof:
    ax[0].plot(p["rr"].r, p["rr"].Gr, color=cm(norm(p["dist"])), lw=1.2)
ax[0].axhline(0, color="0.8", lw=0.8); ax[0].set_xlim(0, 6)
ax[0].set_xlabel("r (Å)"); ax[0].set_ylabel("G(r)")
ax[0].set_title(f"RDF vs distance from interface ({xunit})")
sm = plt.cm.ScalarMappable(cmap=cm, norm=norm); sm.set_array([])
fig.colorbar(sm, ax=ax[0], label=f"distance ({xunit})", fraction=0.046)
ax[1].plot(dists, [p["r1"] for p in prof], "o-", label="1st peak r1 (SRO)")
ax[1].plot(dists, [p["r2"] for p in prof], "s-", label="2nd shell r2 (MRO)")
ax[1].set_xlabel(f"distance from interface ({xunit})"); ax[1].set_ylabel("r (Å)")
ax[1].set_title("SRO/MRO position vs distance"); ax[1].legend()
ax[2].plot(dists, [p["fsdp"] for p in prof], "^-", color="darkorange")
ax[2].set_xlabel(f"distance from interface ({xunit})"); ax[2].set_ylabel("FSDP q (1/Å)")
ax[2].set_title("FSDP (MRO) vs distance")
plt.tight_layout(); plt.show()
print("nearest band = interface-like, farthest = bulk-like")

## 2) 계면 폭 vs 온도 (목표 2)

각 온도에서 계면 구조피크의 **폭(width)** 과 **대비(contrast)** 를 봅니다. 온도↑ 에 폭·대비가
**줄어들면** 계면이 사라지는 것.

In [ ]:
def cube_at(T):
    if USE_SYNTHETIC:
        lam = max(0.6, 6.0*(1 - (T-300)/900))          # 온도↑ → 영향길이↓
        return fds.from_array(make_gradient_cube(lam=lam, seed=int(T)), q_per_px=Q_PER_PX)
    # 실데이터: 파일명에 온도
    return None

if USE_SYNTHETIC:
    Ts = [300, 500, 700, 900, 1100]
    cubes = [(T, cube_at(T)) for T in Ts]
else:
    files = sorted(glob.glob(os.path.join(DATA_ROOT, "*.dm4")))
    cubes = [(fds.coordinate_from_name(os.path.splitext(os.path.basename(p))[0]),
              load_cube(p)) for p in files]
    Ts = [t for t, _ in cubes]

widths, contrasts, infos = [], [], []
for T, cb in cubes:
    inf = interface_distance(cb); infos.append((T, cb, inf))
    widths.append(inf["width"]*xscale); contrasts.append(inf["contrast"])
    print(f"  T={T:>6.0f}K  width={inf['width']:.1f} {xunit}  contrast={inf['contrast']:.4f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(Ts, widths, "o-", color="teal"); ax[0].set_xlabel("temperature (K)")
ax[0].set_ylabel(f"interface width ({xunit})"); ax[0].set_title("interface region width vs T")
ax[1].plot(Ts, contrasts, "s-", color="crimson"); ax[1].set_xlabel("temperature (K)")
ax[1].set_ylabel("interface contrast"); ax[1].set_title("interface contrast vs T")
plt.tight_layout(); plt.show()

## 3) 거리별 FSDP·SRO 를 온도별로 (목표 3, 종합)

각 온도에서 **거리별 FSDP(MRO)** 와 **1st peak(SRO)** 를 구해 겹쳐 봅니다. 계면 근처 → 벌크로 갈수록
구조가 어떻게 회복되는지, 그 **영향 범위가 온도에 따라 어떻게 변하는지**가 한 그림에 나옵니다.

In [ ]:
cmT = plt.get_cmap("rainbow"); normT = plt.Normalize(min(Ts), max(Ts))
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for T, cb, inf in infos:
    pr = distance_profiles(cb, inf)
    if not pr: continue
    dd = [p["dist"] for p in pr]; col = cmT(normT(T))
    ax[0].plot(dd, [p["fsdp"] for p in pr], "o-", color=col, lw=1, label=f"{int(T)}K")
    ax[1].plot(dd, [p["r1"] for p in pr], "o-", color=col, lw=1)
ax[0].set_xlabel(f"distance from interface ({xunit})"); ax[0].set_ylabel("FSDP q (1/Å)")
ax[0].set_title("FSDP (MRO) vs distance, per T"); ax[0].legend(fontsize=7, ncol=2)
ax[1].set_xlabel(f"distance from interface ({xunit})"); ax[1].set_ylabel("1st peak r1 (Å)")
ax[1].set_title("SRO vs distance, per T")
plt.tight_layout(); plt.show()

**정리** — 계면을 구조맵으로 찾아 **거리 밴드별 RDF**로 (1) 벌크/계면 구조, (2) 온도별 계면 폭 소멸,
(3) 거리에 따른 SRO(1st peak)·MRO(FSDP·2nd shell)의 회복과 그 온도 의존성을 정량화합니다.
- 계면이 **기울었으면** `interface_distance`를 per-row(각 행의 argmax)로 바꾸세요.
- 계면이 온도에 따라 **이동**하면 각 온도에서 위치를 다시 찾으므로(이미 그렇게 함) 어느 정도 강건합니다.
- `DIST_BINS_PX`·`*_WIN`·`SCAN_STEP_NM`(거리 nm화)를 데이터에 맞게 조절하세요.